
## Week 2: E-Commerce Sales Database (ShopEase)



## STEP 1: Importing Libraries 

In [1]:
import sqlite3
import pandas as pd
import numpy as np
print("All libraries imported successfully")

All libraries imported successfully


## STEP 2: Connect Database

In [2]:
conn = sqlite3.connect('sales_data.db')
cursor = conn.cursor()
print(" Database connection successful")

 Database connection successful


## Step 3: Create Required Tables

### Table 1 Customer table

In [3]:
cursor.execute('''
CREATE TABLE customers ( 
    customer_id   INT           PRIMARY KEY, 
    first_name    VARCHAR(50)   NOT NULL, 
    last_name     VARCHAR(50)   NOT NULL, 
    email         VARCHAR(100)  UNIQUE NOT NULL, 
    city          VARCHAR(50)   NOT NULL, 
    state         VARCHAR(50)   NOT NULL, 
    join_date     DATE          NOT NULL, 
    is_premium    BOOLEAN       DEFAULT FALSE 
)
''')
# Index for filtering by city/state 
cursor.execute('''
CREATE INDEX idx_customers_city ON customers(city)
''')
cursor.execute('''
CREATE INDEX idx_customers_state ON customers(state)
''')
conn.commit()
print(" customers table created successfully")

 customers table created successfully


### Table 2: Products Table

In [4]:
cursor.execute('''
CREATE TABLE products (
    product_id INT PRIMARY KEY,
    product_name VARCHAR(100) NOT NULL,
    category VARCHAR(50) NOT NULL,
    brand VARCHAR(50) NOT NULL,
    unit_price DECIMAL(10,2) NOT NULL CHECK (unit_price > 0),
    stock_qty INT NOT NULL DEFAULT 0 CHECK (stock_qty >= 0)
)
''')
# Index for filtering by category 
cursor.execute('''
    CREATE INDEX idx_products_category ON products(category)
    ''')
conn.commit()
print("products table created successfully ")

products table created successfully 


In [5]:
cursor.execute('''
CREATE TABLE orders (
    order_id INT PRIMARY KEY,
    customer_id INT NOT NULL,
    order_date DATE NOT NULL,
    status VARCHAR(20) NOT NULL DEFAULT 'Pending'
        CHECK (status IN ('Pending','Shipped','Delivered','Cancelled')),
    total_amount DECIMAL(12,2) NOT NULL CHECK (total_amount >= 0),
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
)
''')
# Index for date-based filtering and sorting 
cursor.execute('''
CREATE INDEX idx_orders_date
ON orders(order_date)
''')
cursor.execute('''
CREATE INDEX idx_orders_status
ON orders(status)
''')
conn.commit()
print("orders table created successfully")

orders table created successfully


### Table 4: Order Items Table

In [6]:
cursor.execute('''
CREATE TABLE order_items (
    item_id INT PRIMARY KEY,
    order_id INT NOT NULL,
    product_id INT NOT NULL,
    quantity INT NOT NULL CHECK (quantity > 0),
    unit_price DECIMAL(10,2) NOT NULL CHECK (unit_price > 0),
    discount_pct DECIMAL(5,2) DEFAULT 0 CHECK (discount_pct BETWEEN 0 AND 100),
    FOREIGN KEY (order_id) REFERENCES orders(order_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
    )
''')

conn.commit()
print(" order_items table created successfully!")
print("\n All 4 tables created")

 order_items table created successfully!

 All 4 tables created


### STEP 4: Inserting Data Into Tables

In [7]:
# customers table data
cursor.execute('''
INSERT INTO customers VALUES
(101, 'Aarav', 'Sharma', 'aarav.s@email.com', 'Mumbai', 'Maharashtra', '2024-01-15', TRUE),
(102, 'Priya', 'Patel', 'priya.p@email.com', 'Ahmedabad', 'Gujarat', '2024-02-20', FALSE),
(103, 'Rohan', 'Gupta', 'rohan.g@email.com', 'Delhi', 'Delhi', '2024-03-10', TRUE),
(104, 'Sneha', 'Reddy', 'sneha.r@email.com', 'Hyderabad', 'Telangana', '2024-04-05', FALSE),
(105, 'Vikram', 'Singh', 'vikram.s@email.com', 'Jaipur', 'Rajasthan', '2024-05-12', TRUE),
(106, 'Ananya', 'Iyer', 'ananya.i@email.com', 'Chennai', 'Tamil Nadu', '2024-06-18', FALSE),
(107, 'Karan', 'Mehta', 'karan.m@email.com', 'Pune', 'Maharashtra', '2024-07-22', TRUE),
(108, 'Divya', 'Nair', 'divya.n@email.com', 'Kochi', 'Kerala', '2024-08-30', FALSE)
''')
print("Data inserted")  

Data inserted


In [8]:
cursor.execute('''
INSERT INTO products VALUES
(201, 'Wireless Earbuds', 'Electronics', 'BoAt', 1499.00, 250),
(202, 'Cotton T-Shirt', 'Clothing', 'Levis', 799.00, 500),
(203, 'Smart Watch', 'Electronics', 'Noise', 2999.00, 150),
(204, 'Running Shoes', 'Clothing', 'Nike', 4599.00, 120),
(205, 'Bluetooth Speaker', 'Electronics', 'JBL', 3499.00, 200),
(206, 'Bedsheet Set', 'Home', 'Spaces', 1299.00, 300),
(207, 'Laptop Stand', 'Electronics', 'AmazonBasics', 899.00, 180),
(208, 'Cushion Covers (Set)', 'Home', 'HomeCenter', 599.00, 400)
''')

conn.commit()
print("Data Inserted Into Product Table")

Data Inserted Into Product Table


In [9]:
cursor.execute('''
INSERT INTO orders VALUES
(1001, 101, '2024-08-01', 'Delivered', 4498.00),
(1002, 102, '2024-08-03', 'Delivered', 799.00),
(1003, 103, '2024-08-05', 'Shipped', 7498.00),
(1004, 101, '2024-08-10', 'Delivered', 3499.00),
(1005, 104, '2024-08-12', 'Cancelled', 2999.00),
(1006, 105, '2024-08-15', 'Delivered', 5898.00),
(1007, 106, '2024-08-18', 'Pending', 1299.00),
(1008, 103, '2024-08-20', 'Delivered', 899.00),
(1009, 107, '2024-08-25', 'Shipped', 6098.00),
(1010, 108, '2024-08-28', 'Delivered', 1598.00)
''')
conn.commit()
print("Data Inserted Into orders Table")

Data Inserted Into orders Table


In [10]:
cursor.execute('''
INSERT INTO order_items VALUES
(5001, 1001, 201, 2, 1499.00, 0),
(5002, 1001, 207, 1, 899.00, 10),
(5003, 1002, 202, 1, 799.00, 0),
(5004, 1003, 203, 1, 2999.00, 0),
(5005, 1003, 204, 1, 4599.00, 5),
(5006, 1004, 205, 1, 3499.00, 0),
(5007, 1005, 203, 1, 2999.00, 0),
(5008, 1006, 201, 1, 1499.00, 10),
(5009, 1006, 204, 1, 4599.00, 5),
(5010, 1007, 206, 1, 1299.00, 0),
(5011, 1008, 207, 1, 899.00, 0),
(5012, 1009, 205, 1, 3499.00, 0),
(5013, 1009, 208, 2, 599.00, 15),
(5014, 1010, 206, 1, 1299.00, 0),
(5015, 1010, 208, 1, 599.00, 0)
''')

conn.commit()
print("Data Inserted Into Orders_Items")

Data Inserted Into Orders_Items


## Query Run function

In [12]:
def run_query(query):
    return pd.read_sql_query(query, conn)
print("Function created successfully")

Function created successfully


## Section A — SQL Basics (SELECT, Constraints, Primary Keys) 

In [13]:
# Q1 Write a query to display all columns and rows from the customer's table. 
q1 = "SELECT * FROM customers"
df_q1 = run_query(q1)
df_q1.style.hide(axis="index")

customer_id,first_name,last_name,email,city,state,join_date,is_premium
101,Aarav,Sharma,aarav.s@email.com,Mumbai,Maharashtra,2024-01-15,1
102,Priya,Patel,priya.p@email.com,Ahmedabad,Gujarat,2024-02-20,0
103,Rohan,Gupta,rohan.g@email.com,Delhi,Delhi,2024-03-10,1
104,Sneha,Reddy,sneha.r@email.com,Hyderabad,Telangana,2024-04-05,0
105,Vikram,Singh,vikram.s@email.com,Jaipur,Rajasthan,2024-05-12,1
106,Ananya,Iyer,ananya.i@email.com,Chennai,Tamil Nadu,2024-06-18,0
107,Karan,Mehta,karan.m@email.com,Pune,Maharashtra,2024-07-22,1
108,Divya,Nair,divya.n@email.com,Kochi,Kerala,2024-08-30,0


In [143]:
# Q2 Retrieve only the first_name, last_name, and city of all customers. 
q2="Select first_name,last_name,city from customers"
df_q2=run_query(q2)
df_q2.style.hide(axis="index")

first_name,last_name,city
Aarav,Sharma,Mumbai
Priya,Patel,Ahmedabad
Rohan,Gupta,Delhi
Sneha,Reddy,Hyderabad
Vikram,Singh,Jaipur
Ananya,Iyer,Chennai
Karan,Mehta,Pune
Divya,Nair,Kochi


In [15]:
# Q3 List all unique categories available in the products table. 
q3="SELECT DISTINCT category from products"
df_q3=run_query(q3)
df_q3

,category
0,Clothing
1,Electronics
2,Home


In [16]:
#Q4 Identify the Primary Key of each table in the schema. Explain why a Primary Key must be unique and NOT NULL
d1=run_query("PRAGMA table_info(customers)")
print(f"Primary key of customers table:{d1[d1["pk"]==1]["name"].values}")
d2=run_query("PRAGMA table_info(products)")
print(f"Primary key of product table:{d2[d2["pk"]==1]["name"].values}")
d3=run_query("PRAGMA table_info(orders)")
print(f"Primary key of orders table:{d3[d3["pk"]==1]["name"].values}")
d4=run_query("PRAGMA table_info(orders)")
print(f"Primary key of order_item table:{d4[d4["pk"]==1]["name"].values}")


Primary key of customers table:['customer_id']
Primary key of product table:['product_id']
Primary key of orders table:['order_id']
Primary key of order_item table:['order_id']


In [76]:
#Q5. What constraints are applied to the email column in the customers table? What would happen if you tried to insert a duplicate email? 
print("email is defined as:email VARCHAR(100) UNIQUE NOT NULL")
print("""constraints are:
1:UNIQUE Constraint
2:NOT NULL Constraint
""")
print("""If an email address that already exists in the table is inserted
the database will reject the operation and return a UNIQUE constraint violation error.""")


email is defined as:email VARCHAR(100) UNIQUE NOT NULL
constraints are:
1:UNIQUE Constraint
2:NOT NULL Constraint

If an email address that already exists in the table is inserted
the database will reject the operation and return a UNIQUE constraint violation error.


In [79]:
# Q6. Try inserting a product with unit_price = -50. What happens and which constraint prevents it? Write both 
# the INSERT statement and explain the error. 
print("unit_price is defined as:unit_price DECIMAL(10,2) NOT NULL CHECK (unit_price > 0)")
print("""
The CHECK constraint (unit_price > 0) ensures that the price of a product must be greater than zero
""")
print("""
When the above INSERT statement is executed
the database rejects the row because -50 does
not satisfy the condition unit_price > 0""")

unit_price is defined as:unit_price DECIMAL(10,2) NOT NULL CHECK (unit_price > 0)

The CHECK constraint (unit_price > 0) ensures that the price of a product must be greater than zero


When the above INSERT statement is executed
the database rejects the row because -50 does
not satisfy the condition unit_price > 0


## Section B — Filtering & Optimization (WHERE, Indexes)

In [18]:
# Q7 Retrieve all orders with status = 'Delivered'.
q5="SELECT * FROM orders WHERE status = 'Delivered'"
df_q5=run_query(q5)
df_q5

,order_id,customer_id,order_date,status,total_amount
0,1001,101,2024-08-01,Delivered,4498
1,1002,102,2024-08-03,Delivered,799
2,1004,101,2024-08-10,Delivered,3499
3,1006,105,2024-08-15,Delivered,5898
4,1008,103,2024-08-20,Delivered,899
5,1010,108,2024-08-28,Delivered,1598


In [19]:
# Q8 Find all products in the 'Electronics' category with a unit_price greater than ₹2000.
q6="select * from products where category='Electronics' and unit_price>2000"
df_q6=run_query(q6)
df_q6

,product_id,product_name,category,brand,unit_price,stock_qty
0,203,Smart Watch,Electronics,Noise,2999,150
1,205,Bluetooth Speaker,Electronics,JBL,3499,200


In [20]:
# Q9 List all customers who joined in the year 2024 and belong to the state 'Maharashtra'.
q7="select *from customers where strftime('%Y', join_date) = '2024' and state='Maharashtra' "
df_q7=run_query(q7)
df_q7

,customer_id,first_name,last_name,email,city,state,join_date,is_premium
0,101,Aarav,Sharma,aarav.s@email.com,Mumbai,Maharashtra,2024-01-15,1
1,107,Karan,Mehta,karan.m@email.com,Pune,Maharashtra,2024-07-22,1


In [21]:
# Q10 Find all orders placed between '2024-08-10' and '2024-08-25' (inclusive) that are NOT cancelled.
q8 = "Select * FROM orders WHERE order_date BETWEEN '2024-08-10' and '2024-08-25' and status != 'Cancelled'"
df_q8=run_query(q8)
df_q8.style.hide(axis="index")

order_id,customer_id,order_date,status,total_amount
1004,101,2024-08-10,Delivered,3499
1006,105,2024-08-15,Delivered,5898
1007,106,2024-08-18,Pending,1299
1008,103,2024-08-20,Delivered,899
1009,107,2024-08-25,Shipped,6098


In [89]:
# Q11 Explain what the index idx_orders_date does. How would it improve the performance of a query that 
# filters orders by order_date? Write a sample query that would benefit from this index
print("The index is:CREATE INDEX idx_orders_date ON orders(order_date);")
print("""
The idx_orders_date index creates a separate data structure that stores the values of the order_date 
column in sorted order along with references to the corresponding rows in the orders table
""")
print("""
How does it improve performance?\n
The database can directly access to the required date values
Fewer rows need to be search
Queries that search by order_date execute much faster
""")

The index is:CREATE INDEX idx_orders_date ON orders(order_date);

The idx_orders_date index creates a separate data structure that stores the values of the order_date 
column in sorted order along with references to the corresponding rows in the orders table


How does it improve performance?

The database can directly access to the required date values
Fewer rows need to be search
Queries that search by order_date execute much faster



In [93]:
print("Sample query that would benefit from this index ")
run_query("""select *
from orders
where order_date = '2024-08-20'""")

Sample query that would benefit from this index 


,order_id,customer_id,order_date,status,total_amount
0,1008,103,2024-08-20,Delivered,899


In [107]:
# Q12. If you run: SELECT * FROM customers WHERE YEAR(join_date) = 2024; — would the index on join_date 
# be used? Explain why or why not, and rewrite the query to be index-friendly (SARGable). 

print("""If we run this query:\nSELECT * 
FROM customers 
WHERE YEAR(join_date) = 2024""")
print()
print("The index on join_date would generally not be used efficiently")
print("""\nBecause the expression:(YEAR(join_date)\n applies YEAR function to every value in the 
join_date column before comparing it with 2024""")
print()
print("Index friendly query:")
print("""select *
from customers
where join_date >= '2024-01-01'
  and join_date < '2025-01-01'""")
run_query("""select *
from customers
where join_date >= '2024-01-01'
  and join_date < '2025-01-01'""").head(3)

If we run this query:
SELECT * 
FROM customers 
WHERE YEAR(join_date) = 2024

The index on join_date would generally not be used efficiently

Because the expression:(YEAR(join_date)
 applies YEAR function to every value in the 
join_date column before comparing it with 2024

Index friendly query:
select *
from customers
where join_date >= '2024-01-01'
  and join_date < '2025-01-01'


,customer_id,first_name,last_name,email,city,state,join_date,is_premium
0,101,Aarav,Sharma,aarav.s@email.com,Mumbai,Maharashtra,2024-01-15,1
1,102,Priya,Patel,priya.p@email.com,Ahmedabad,Gujarat,2024-02-20,0
2,103,Rohan,Gupta,rohan.g@email.com,Delhi,Delhi,2024-03-10,1


## Section C — Aggregation (GROUP BY, SUM, COUNT, AVG, MIN, MAX)

In [22]:
# Q13 Count the total number of orders in the orders table. 
q9="select count(*) as total_orders from orders"
df_q9=run_query(q9)
df_q9.style.hide(axis="index")

total_orders
10


In [23]:
# Q14 Find the total revenue (SUM of total_amount) from all 'Delivered' orders.
q10="select sum(total_amount) as total_revenue_delivered from orders where status='Delivered'"
df_10=run_query(q10)
df_10

,total_revenue_delivered
0,17191


In [24]:
# Q15 Calculate the average unit_price of products in each category.
q11="SELECT category, AVG(unit_price) as avg_price from products group by category"
df_q11=run_query(q11)
df_q11

,category,avg_price
0,Clothing,2699.0
1,Electronics,2224.0
2,Home,949.0


In [25]:
# Q16. For each order status, find the count of orders and the total revenue. Sort the result by total revenue in descending order. 
q12="""select status ,count(*) as order_count,sum(total_amount) as total_revenue  from orders
group by status order by total_revenue desc
"""
df_q12=run_query(q12)
df_q12

,status,order_count,total_revenue
0,Delivered,6,17191
1,Shipped,2,13596
2,Cancelled,1,2999
3,Pending,1,1299


In [26]:
# Q17 Find the most expensive (MAX) and cheapest (MIN) product in each category.
q12="""select category, max(unit_price),min(unit_price) from products
group by category """
df_q12=run_query(q12)
df_q12.style.hide(axis="index")

category,max(unit_price),min(unit_price)
Clothing,4599,799
Electronics,3499,899
Home,1299,599


In [27]:
# Q18. List all product categories where the average unit_price is greater than ₹2000. (Hint: Use HAVING clause)
q13="""select category, avg(unit_price) as average_unit_price from products
group by category 
having average_unit_price>2000"""
df_q13=run_query(q13)
df_q13

,category,average_unit_price
0,Clothing,2699.0
1,Electronics,2224.0


 ## Section D — Joins & Relationships 

In [28]:
# Q19. Write an INNER JOIN query to display each order along with the customer's first_name and last_name. 
# Show: order_id, order_date, first_name, last_name, total_amount. 
q14="""select o.order_id, o.order_date, c.first_name, c.last_name, o.total_amount 
from orders o 
INNER JOIN customers c ON o.customer_id = c.customer_id 
order by o.order_id"""
df_q14=run_query(q14)
df_q14

,order_id,order_date,first_name,last_name,total_amount
0,1001,2024-08-01,Aarav,Sharma,4498
1,1002,2024-08-03,Priya,Patel,799
2,1003,2024-08-05,Rohan,Gupta,7498
3,1004,2024-08-10,Aarav,Sharma,3499
4,1005,2024-08-12,Sneha,Reddy,2999
5,1006,2024-08-15,Vikram,Singh,5898
6,1007,2024-08-18,Ananya,Iyer,1299
7,1008,2024-08-20,Rohan,Gupta,899
8,1009,2024-08-25,Karan,Mehta,6098
9,1010,2024-08-28,Divya,Nair,1598


In [29]:
# Q20. Using a LEFT JOIN, list ALL customers and their orders (if any). Customers with no orders should still 
# appear with NULL values for order columns. 
q15="""select c.customer_id, c.first_name, c.last_name, 
       o.order_id, o.order_date, o.total_amount 
from customers c 
LEFT JOIN orders o ON c.customer_id = o.customer_id 
order by c.customer_id """
df_15=run_query(q15)
df_15

,customer_id,first_name,last_name,order_id,order_date,total_amount
0,101,Aarav,Sharma,1001,2024-08-01,4498
1,101,Aarav,Sharma,1004,2024-08-10,3499
2,102,Priya,Patel,1002,2024-08-03,799
3,103,Rohan,Gupta,1003,2024-08-05,7498
4,103,Rohan,Gupta,1008,2024-08-20,899
5,104,Sneha,Reddy,1005,2024-08-12,2999
6,105,Vikram,Singh,1006,2024-08-15,5898
7,106,Ananya,Iyer,1007,2024-08-18,1299
8,107,Karan,Mehta,1009,2024-08-25,6098
9,108,Divya,Nair,1010,2024-08-28,1598


In [30]:
# Q21. Write a query using JOINs across three tables (orders → order_items → products) to show: order_id, 
# product_name, quantity, unit_price, and discount_pct for each order item.
q16= """select oi.order_id, p.product_name, oi.quantity, 
       oi.unit_price, oi.discount_pct 
from orders o 
INNER JOIN order_items oi ON o.order_id = oi.order_id 
INNER JOIN products p ON oi.product_id = p.product_id 
order by oi.order_id"""
df_q16=run_query(q16)
df_q16

,order_id,product_name,quantity,unit_price,discount_pct
0,1001,Wireless Earbuds,2,1499,0
1,1001,Laptop Stand,1,899,10
2,1002,Cotton T-Shirt,1,799,0
3,1003,Smart Watch,1,2999,0
4,1003,Running Shoes,1,4599,5
5,1004,Bluetooth Speaker,1,3499,0
6,1005,Smart Watch,1,2999,0
7,1006,Wireless Earbuds,1,1499,10
8,1006,Running Shoes,1,4599,5
9,1007,Bedsheet Set,1,1299,0


In [116]:
# Q22. Explain the difference between LEFT JOIN and RIGHT JOIN with an example from this schema. When 
# would you use a FULL OUTER JOIN?
print("""
1. LEFT JOIN

 LEFT JOIN returns:

All rows from the left table
Matching rows from the right table
NULL values for columns from the right table when no match exists""")
print("""
2. RIGHT JOIN

 RIGHT JOIN returns:

All rows from the right table
Matching rows from the left table
NULL values for columns from the left table when no match exists""")
print()
print("FULL OUTER JOIN use when you want to find:")
print("""
Customers who have never placed an order.
Orders that do not have a matching customer""")


1. LEFT JOIN

 LEFT JOIN returns:

All rows from the left table
Matching rows from the right table
NULL values for columns from the right table when no match exists

2. RIGHT JOIN

 RIGHT JOIN returns:

All rows from the right table
Matching rows from the left table
NULL values for columns from the left table when no match exists

FULL OUTER JOIN use when you want to find::

Customers who have never placed an order.
Orders that do not have a matching customer


In [142]:
# Q23. Identify all Foreign Key relationships in the schema. Explain what would happen if you tried to insert an 
# order with customer_id = 999 (which doesn't exist in customers)
print("""
All foreign keys in schema are:\n
FOREIGN KEY (customer_id)
REFERENCES customers(customer_id)
FOREIGN KEY (order_id)
REFERENCES orders(order_id)
FOREIGN KEY (product_id)
REFERENCES products(product_id)

""")
print("""If I tried to insert an order with customer_id = 999 then:\n
The database will reject the insertion and raise a Foreign Key 
Constraint Violation error because the value 999 is not present in the parent table (customers)
""")


All foreign keys in schema are:

FOREIGN KEY (customer_id)
REFERENCES customers(customer_id)
FOREIGN KEY (order_id)
REFERENCES orders(order_id)
FOREIGN KEY (product_id)
REFERENCES products(product_id)


If I tried to insert an order with customer_id = 999 then:

The database will reject the insertion and raise a Foreign Key 
Constraint Violation error because the value 999 is not present in the parent table (customers)



## Section E — Advanced Concepts (CASE, ACID, Transactions) 

In [144]:
# Q24. Write a query using CASE to classify products into price tiers: 
# • 'Budget'    → unit_price < 1000 
# 'Mid-Range' → unit_price BETWEEN 1000 AND 3000 
# 'Premium'   → unit_price > 3000 
# Display: product_name, unit_price, price_tier.

q17="""select product_name, unit_price,
       case
           when unit_price < 1000 THEN 'Budget'
           when unit_price BETWEEN 1000 AND 3000 THEN 'Mid-Range'
           when unit_price > 3000 THEN 'Premium'
       end as price_tier
from products 
order by unit_price"""
df_q17=run_query(q17)
df_q17.style.hide(axis="index")

product_name,unit_price,price_tier
Cushion Covers (Set),599,Budget
Cotton T-Shirt,799,Budget
Laptop Stand,899,Budget
Bedsheet Set,1299,Mid-Range
Wireless Earbuds,1499,Mid-Range
Smart Watch,2999,Mid-Range
Bluetooth Speaker,3499,Premium
Running Shoes,4599,Premium


In [32]:
# Q25. Using a CASE statement inside an aggregate function, count how many orders are 'Delivered' vs 'Not 
# Delivered' (all other statuses). Display the result in a single row. 
q18="""select
    sum(case when status = 'Delivered' then 1 ELSE 0 END) as delivered_count,
    sum(case when status != 'Delivered' then 1 ELSE 0 END) as not_delivered_count
from orders"""
df_18=run_query(q18)
df_18.style.hide(axis="index")

delivered_count,not_delivered_count
6,4


In [145]:
# Q26. Explain each letter of ACID: 
# • A – Atomicity 
# • C – Consistency 
# •I – Isolation 
# •D – Durability 
# Give a real-world example (e.g., bank transfer) showing why each property is important.
print("""ACID is a set of properties that ensure database 
transactions are processed reliably and maintain data integrity\n""")
print("""
1->Atomicity:Atomicity means a transaction is treated as a single unit of work, Either
all operations succeed or none of them are applied\n
2->Consistency=Consistency ensures that a transaction should goes from one valid
state to another valid state while following constraints
\n
3->Isolation:Isolation ensures that multiple transactions running at the same time do not interfere with each other\n
4->Durability:Durability ensures that once a transaction is committed, the changes are permanently 
saved, even if the system crashes immediately """)


ACID is a set of properties that ensure database 
transactions are processed reliably and maintain data integrity


1->Atomicity:Atomicity means a transaction is treated as a single unit of work, Either
all operations succeed or none of them are applied

2->Consistency=Consistency ensures that a transaction should goes from one valid
state to another valid state while following constraints


3->Isolation:Isolation ensures that multiple transactions running at the same time do not interfere with each other

4->Durability:Durability ensures that once a transaction is committed, the changes are permanently 
saved, even if the system crashes immediately 


In [148]:
print("Example of Transaction:")
print("""
 Atomicity
If ₹5,000 is transferred from A to B, if 2 steps are complete and last step is not complete then 
the whole transaction is rolled back

Consistency 
Before transfer: Total money = ₹13,000
After transfer: Total money = ₹8000
The database remains correct and consistence

Isolation 
If two users did transaction simultaneously, then transactions works independently

Durability
Once the transfer is completed and committed, it remains saved even if the bank server crashes
""")

Example of Transaction:

 Atomicity
If ₹5,000 is transferred from A to B, if 2 steps are complete and last step is not complete then 
the whole transaction is rolled back

Consistency 
Before transfer: Total money = ₹13,000
After transfer: Total money = ₹8000
The database remains correct and consistence

Isolation 
If two users did transaction simultaneously, then transactions works independently

Durability
Once the transfer is completed and committed, it remains saved even if the bank server crashes



In [147]:
# Q27. Write a SQL transaction that does the following atomically: 
# 1. Insert a new order (order_id=1011, customer_id=102, today's date, 'Pending', 1598.00) 
# 2. Insert two order items for that order 
# 3. Update the stock_qty of the purchased products 
# 4. If any step fails, ROLLBACK the entire transaction. Otherwise, COMMIT. 
# Write the complete BEGIN...COMMIT/ROLLBACK block.
print("SQL Transaction Example")
print("""
Example:
Customer with customer_id (102) places new order
Item:
Smart Watch=(product_id=203, qtn=1, unit_price=2999)
Bedsheet Set=(product_id=206, qtn=1, unit_price=1299)
Total_amount=5597.0""")
print()
print()
print("Executing Transaction")
try:
    print("Begin Transaction")
     # Insert order
    print("Insert new order order_id(1011)")
    cursor.execute("""insert into orders values 
                    (1011, 102, '2026-05-30', 'Pending', 5597.00)""")
    print("order placed successfully")
     # Insert order items
    print("Insert into order_items")
    cursor.execute("""insert into order_items values
                    (5016, 1011, 203, 1, 2999.00, 0)""")
    print("Item 1: Smart watch")
    cursor.execute("""insert into order_items values
                    (5017, 1011, 206, 2, 1299.00, 0)""")
    print("  Item 2: Cushion Covers")

     # Update stock
    print(" Update product stock quantities")
    cursor.execute("update products set stock_qty = stock_qty - 1 where product_id = 203")
    print("  Smart Watch stock: -1")
    
    cursor.execute("update products set stock_qty = stock_qty - 2 where product_id = 206")
    print("  Bedsheet Set stock: -2")       

    # COMMIT
    print("  COMMIT transaction")
    conn.commit()
    print("TRANSACTION SUCCESSFUL")
except Exception as e:
    print(f"{e}")
    print("\nROLLBACK: Undo all changes ")
    conn.rollback()                      
    print(" Transaction rolled back - no changes made")

SQL Transaction Example

Example:
Customer with customer_id (102) places new order
Item:
Smart Watch=(product_id=203, qtn=1, unit_price=2999)
Bedsheet Set=(product_id=206, qtn=1, unit_price=1299)
Total_amount=5597.0


Executing Transaction
Begin Transaction
Insert new order order_id(1011)
UNIQUE constraint failed: orders.order_id

ROLLBACK: Undo all changes 
 Transaction rolled back - no changes made


### Verify Transaction Results

In [69]:

print(" Verification: New Order")
query_verify_order = "select * from orders where order_id = 1011"
df_verify = run_query(query_verify_order)
df_verify





 Verification: New Order


,order_id,customer_id,order_date,status,total_amount
0,1011,102,2026-05-30,Pending,5597


In [70]:
print("Verification: Order Items")
query_verify_items = """select oi.item_id, oi.order_id, p.product_name, 
                         oi.quantity, oi.unit_price 
                  from order_items oi 
                  inner join products p on oi.product_id = p.product_id 
                  where oi.order_id = 1011"""
df_items = run_query(query_verify_items)
df_items

Verification: Order Items


,item_id,order_id,product_name,quantity,unit_price
0,5016,1011,Smart Watch,1,2999
1,5017,1011,Bedsheet Set,2,1299


In [72]:
print(" Verification: Updated Stock")
query_stock = """select product_id, product_name, stock_qty 
                 from products where product_id in (203, 206)"""
df_stock = run_query(query_stock)
df_stock

 Verification: Updated Stock


,product_id,product_name,stock_qty
0,203,Smart Watch,149
1,206,Bedsheet Set,298
